# spinpy en cinco pasos

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cgt1989/spinpy/blob/main/ejemplos/spinpy_inicio.ipynb)

Este cuaderno recorre el flujo completo sobre un VOI porcino abierto
(Koria, Mengoni y Brockett 2020, CC BY 4.0): leer, recortar un cubo y un
cilindro, medir la morfometría y ensayar a compresión. No necesita datos
propios ni instalar nada en tu equipo. Tarda unos dos minutos.

Los números mecánicos son resultados de simulación con tejido homogéneo e
isótropo, verificados numéricamente; no son predicciones validadas contra un
ensayo físico de este hueso.

## 1. Instalar

In [ ]:
import os, subprocess, sys
# En Colab se clona el repositorio (trae los VOIs de ejemplo); si ya estas
# dentro de el, se usa tal cual.
if not os.path.exists("pyproject.toml"):
    if not os.path.exists("spinpy-repo"):
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/cgt1989/spinpy.git", "spinpy-repo"], check=True)
    os.chdir("spinpy-repo")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[elastico]", "tifffile"], check=True)

## 2. Leer el VOI

In [ ]:
import numpy as np
import spinpy
from spinpy import leer_voi

BW, sp = leer_voi("datos/porcino_koria2020/VOI_V1.mat")
print(f"spinpy {spinpy.__version__} | VOI {BW.shape}, voxel {sp[0]*1000:.1f} µm, BV/TV {BW.mean():.3f}")

## 3. Recortar: cubo o cilindro

`recortar` es el mismo punto de entrada que usa la ventana de recorte de la
interfaz. Con un cilindro devuelve la caja que lo envuelve y su **dominio**: la
mascara de los voxeles que pertenecen a él. Toda cifra medida sobre un cilindro
debe restringirse al dominio; si se mide la caja entera, el exterior cuenta como
poro y el BV/TV sale unas π/4 veces el real.

In [ ]:
from spinpy import recortar

centro = np.array(BW.shape) * sp[0] / 2          # el centro del VOI, en mm
ejes, origen = np.eye(3), np.zeros(3)            # sin reorientar por PCA

cubo = recortar(BW, sp, ejes, origen,
                {"forma": "cubo", "centro_pca": centro, "lado_mm": 1.5})
cil = recortar(BW, sp, ejes, origen,
               {"forma": "cilindro", "centro_pca": centro,
                "radio_mm": 0.75, "altura_mm": 1.5, "eje": 2})

print("cubo     ", cubo["cubo"].shape, f"BV/TV {cubo['bvtv']:.3f}")
print("cilindro ", cil["cubo"].shape, f"BV/TV {cil['bvtv']:.3f}",
      f"(la caja entera daría {cil['cubo'].mean():.3f}: no es el valor del cilindro)")

## 4. Morfometría

In [ ]:
from spinpy import morfometria, morfometria_dominio

m_cubo = morfometria(cubo["cubo"], sp)
m_cil = morfometria_dominio(cil["cubo"], cil["dominio"], sp)   # solo dentro del cilindro

for k, u in (("BVTV", ""), ("TbTh", "mm"), ("TbSp", "mm"), ("TbN", "1/mm")):
    print(f"{k:5s} cubo {m_cubo[k]:8.4f}   cilindro {m_cil[k]:8.4f}  {u}")
print("DA (solo cubo):", round(m_cubo["DA"], 3))

## 5. Ensayo de compresión del cubo

In [ ]:
from spinpy import criterio_pistoia, ensayo_compresion

lado = 40
c = cubo["cubo"]
sub = c[:lado, :lado, :lado]
r = ensayo_compresion(sub, sp, E_s=20e9, nu_s=0.3, sigma0=1e6, apoyo="deslizante")
p = criterio_pistoia(r)
print(f"E aparente {r['E_app']/1e6:.0f} MPa | carga de fallo (Pistoia) {p['sigma_fallo']/1e6:.2f} MPa")

## Y ahora

- Interfaz gráfica completa: `python visor.py` (en tu equipo, con `pip install -e ".[gui]"`).
- Más motores FEM: `pip install -e ".[fem]"` y `python -m spinpy.fem --help`.
- Mapa del código y cómo contribuir: `docs/ARQUITECTURA.md` y `CONTRIBUTING.md`.